# STAT 301 Project — Online Shoppers Purchasing Intention

**Name:** Ivy Yan   
**Group:**  group-25   
**Student number:** 39512314 

---


## Section 0: TA Feedback

#### Score 24.4/30
Mechanics: 3/3

Reasoning: 9/12

-Feedback: incorrect use of causal implication in scientific question.  
I use **`highlight`** to correct.

Writing: 2.4/3

-Feedback: need more comments in code.

Visualization: 10/12

-Feedback: visualization hard to read - change y limit since average exit rate is non-negative.  

## Section 1: Data Description


### 1. Descriptive Summary

#### Data description

This data contains information about user behavior when online shopping. Each row in the dataset represents one online shopping session, and the variables describe various aspects of user behavior, technical attributes, and timing information.  
The dataset includes both quantitative and categorical variables, such as the number and duration of pages visited, bounce and exit rates, month of visit, type of operating system, and whether the session occurred on a weekend.  

- **Number of observations:** 12330  
- **Number of variables:** 18
#### Variables description
  | Variable Name | Type | Description |
|-----------------------------|------------------|-------------------------------------------------------------|
| Administrative |Numeric(Integer) | Number of administrative pages visited by the user. |
| Administrative_Duration | Numeric(Integer) | Total time spent on administrative-related pages (in seconds). |
| Informational | Numeric(Integer) | Number of informational pages visited by the user. |
| Informational_Duration | Numeric(Integer) | Total time spent on informational pages (in seconds). |
| ProductRelated | Numeric(Integer) | Number of product-related pages visited by the user. |
| ProductRelated_Duration | Numeric(Continuous) | Total time spent on product-related pages (in seconds). |
| BounceRates | Numeric(Continuous) | percentage of visitors who enter the site from that page and then leave ("bounce") without triggering any other requests to the analytics server during that session. |
| ExitRates | Numeric(Continuous) | percentage of this page being the last session. |
| PageValues | Numeric(Integer) | Average value for a web page that a user visited before completing an e-commerce transaction.  |
| SpecialDay | Numeric(Integer) | Closeness of the site visit date to a special day (e.g., Mother’s Day, Christmas). Ranges from 0 to 1. |
| Month | Categorical | Month of the visit (Feb–Dec). |
| OperatingSystems | Categorical | Type of operating system used by the visitor (e.g., Windows, Mac). |
| Browser | Categorical | Type of browser used (e.g., Chrome, Firefox, Safari). |
| Region | Categorical | Geographic region of the visitor. |
| TrafficType | Categorical | Type of traffic source (e.g., direct, referral, search). |
| VisitorType | Categorical | Indicates if the visitor is a Returning Visitor or New Visitor. |
| Weekend | Binary (True/False) | Indicates whether the visit occurred on a weekend. |
| Revenue | Binary (0/1) | Indicates whether the session ended with a purchase (1 = purchase, 0 = no purchase). |

---



### 2. Source and Information 
This dataset is the **Online Shoppers Purchasing Intention** dataset published by UCI Machine Learning Repository. 
It contains session-level features for e-commerce website visits and a binary outcome `Revenue` indicating whether a purchase occurred. 



### 3. Pre-selection of Variables 
I will first drop `Administrative_Duration`, `Informational_Duration` and `ProductRelated_Duration` because they are **redundant**. They are highly associated with `Administrative`, `Informational` and `ProductRelated`, which may cause multicollinearity. `OperatingSystems`, `browser`, and `region` will also be deleted because they don't have a clear categorization.

Other variables will be selected to further analyse.


## Section 2: Scientific Question


### 1. Scientific Question 
**Question:** _How do the number of product-related pages viewed (ProductRelated), the proximity of the session to a special day (SpecialDay), and the exit rates of visited pages (ExitRates) **`associate with`** the likelihood that an online shopping session ends with a purchase (Revenue)?_

### 2. Name the Response 
**Response:** `Revenue` (binary: 1 if the session resulted in a purchase, 0 otherwise).

### 3. Aim: Prediction, Inference, or Both? 
It is an **Inference** model, because we aim to understand how productRelated, exit rates, and special day — are associated with the purchase outcome (`Revenue`).


## Section 3: Exploratory Data Analysis and Visualization (EDA)

In [ ]:
library(tidyverse)
library(repr)
library(infer)
library(broom)
library(dplyr)
library(ggplot2)
library(car)

In [ ]:
#Read the data
online_shopping_raw <- read.csv("online_shoppers_intention.csv")

#Clean the data
online_shopping <- online_shopping_raw %>%
filter(is.na(Region) | Region != 1) |>  #Remove Region = 1
mutate( Revenue = factor(Revenue, levels = c(TRUE, FALSE),
                     labels = c("Purchase", "No Purchase")), #Recode Revenue to "Purchase" and "No Purchase"
    NearSpecialDay = if_else(SpecialDay > 0, "Near Special Day", "Regular Day")) #Create NearSpecialDay indicator from SpecialDay

head(online_shopping)
nrow(online_shopping)

In [ ]:
# Fit Data into glm model
online_shopping_glm <- glm(Revenue ~ SpecialDay + ProductRelated + ExitRates, data = online_shopping, family = 'binomial')

# Get model summary in log-odds scale with 95% CI
online_shopping_model <- tidy (online_shopping_glm, conf.int = TRUE, conf.level = 0.95)
online_shopping_model

# Get model summary in odds ratio scale (exponentiated coefficients)
online_shopping_model_odds <- tidy (online_shopping_glm, conf.int = TRUE, conf.level = 0.95, exponentiate = TRUE)
online_shopping_model_odds 

In [ ]:
# Plot1: ProductRelated vs Revenue
p1 <- ggplot(online_shopping, aes(x = factor(Revenue), y = ProductRelated, fill = factor(Revenue))) +
  geom_boxplot(width = 0.15, alpha = 0.8) +
  labs(x = "Revenue (0/1)", y = "ProductRelated") +
  theme_minimal() +
  theme(legend.position = "none")

# Plot2: ExitRates vs Revenue
p2 <- ggplot(online_shopping, aes(x = factor(Revenue), y = ExitRates, fill = factor(Revenue))) +
  geom_boxplot(width = 0.15, alpha = 0.8) +
  labs(x = "Revenue (0/1)", y = "ExitRates") +
  theme_minimal() +
  theme(legend.position = "none")

# Plot3: SpecialDay vs Revenue
p3 <- ggplot(online_shopping, 
       aes(x = NearSpecialDay, fill = Revenue)) +
  geom_bar(position = "fill") +
  scale_y_continuous(labels = scales::percent) +
  labs(x = "Special Day Category",
       y = "Proportion of Purchase / No Purchase",
       fill = "Revenue") +
  theme_minimal()

# Plot4: Logestic curve for ExitRates
p4 <- ggplot(online_shopping, aes(x = ExitRates, y = Revenue)) +
  geom_jitter(height = 0.05, alpha = 0.2) +
  geom_smooth(method = "glm",
              method.args = list(family = "binomial"),
              se = TRUE) +
  labs(x = "ExitRates", y = "Pr(Revenue = 1)") +
  theme_minimal()

# Combine all plots
library(patchwork)

(p1 | p2) /
(p3 | p4)


### Interpretations 

#### 1. Why this plot?
This visualization plot puts all variables together. There are different colors to separate our level of the binary response variable, so we can clearly see the relationship between variables.

#### 2. Brief results:
_We set `No Purchase` as the reference level. There is a negative association between the Exit rate and making a purchase. Product-related pages have a positive association with making a purchase. When near special days, this behavior becomes more obvious, meaning there is an association between Special days and purchase behavior._

#### 3. What we learn / potential issues:
One big potential issue is that we product-related data is really right-tailed, so a few extreme sessions can distort linear fits. Besides, there is a big overlap between `Purchase` and `No Purchase`, making it hard to distinguish.

## Section 4: Method and Plan

#### 1. Why this method
We use the logistic regression method because our response variable, `Revenue` is binary. The Logistic function can ensure that our prediction output always falls within the [0,1] range.

#### 2. Assumption required
This method assumes that: 
1. Observations are **independent**.  
2. The log-odds of the outcome are **linearly related** to the explanatory variables.  
3. The explanatory variables are measured without **substantial error**.

#### 3. Limitations
The main limitation of our logistic regression is that it is sensitive to outliers. Besides, it only captures associations, not causal effects, because the dataset is observational rather than experimental.

## Section 5: Computation Code and Outcome

#### 1. Virable selected and Model Fitting
- We first drop the meaningless variables `OperatingSystems` and `Browser`.
- Then fit all variables into the Logistic Model.
- Reduce highly correlated variables using VIF.
- Make the reduced model.

In [ ]:
# Drop OperatingSystems and Browser
Online_Shopping_1 <- online_shopping %>%
select(
    Revenue,
    Administrative, Administrative_Duration,
    Informational, Informational_Duration,
    ProductRelated, ProductRelated_Duration,
    BounceRates, ExitRates, PageValues,
    SpecialDay, NearSpecialDay,
    Month, Region, TrafficType, VisitorType, Weekend)

# Create a Numeric value for Revenue
Online_Shopping_2 <- Online_Shopping_1 %>%
  mutate(Revenue01 = if_else(Revenue == "Purchase", 1, 0))

# Fit all variables into Logistic Model
full_model <- glm(
  Revenue01 ~ Administrative + Informational + ProductRelated +
    Administrative_Duration + Informational_Duration + ProductRelated_Duration +
    BounceRates + ExitRates + PageValues +
    NearSpecialDay + Month + Region + TrafficType + VisitorType + Weekend,
  data = Online_Shopping_2,
  family = binomial
)

# Compute VIF for all predictors
vif(full_model)

# Drop variables with high VIF
tidy_reduced <- tidy(full_model, conf.level = 0.05, conf.int = TRUE)
tidy_reduced


#### 2. Interpretation
1. `ProductRelated` shows a statistically **insignificant association** with purchasing because P> 0.05.
2. `ExitRates` has a significant **negative** association with making a purchase.
3. There's **no** significant association between `SpecialDay` and purchase likelihood because P = 0.677 >0.05.
4. For Other potential predictors,`PageValues` and `WeekendTRUE` have a **positive** association with purchase likelihood, while other variables have **no** significant association with the response variable `Revenue`.


### p